# PA deck deterioration — a Lu & Guler (2022)-style analysis + climate contribution

Follows the outcome definition and covariates of **Lu, M., & Guler, S. I. (2022),
"Comparison of Random Survival Forest with Accelerated Failure Time-Weibull Model for
Bridge Deck Deterioration," Transportation Research Record**
(DOI 10.1177/03611981221078281) on NBI data, then adds Daymet climate +
coastal-distance features. The marginal C-index from climate exposure is this study's
contribution. **This is a similar analysis on public NBI data, not a replication** —
the data source, spell construction, and traffic clock all differ (see below), so
agreement is expected to be qualitative, not numeric.

**Their design:** PA state-owned decks, outcome = time a deck spends at condition rating 6
before dropping to 5, RSF vs AFT-Weibull, 80/20 split. Reported test C-indexes:
RSF 0.836 (cumulative-truck-traffic clock), RSF 0.591 (calendar clock), AFT 0.693 (traffic).

**How the three published numbers compare with this study** (all three reported, not
just the agreeing ones):

| Model, clock | Lu & Guler 2022 | here (NBI spells) | reading |
|---|---|---|---|
| RSF, traffic clock | 0.836 | 0.876 | qualitatively agrees (traffic clock is the strong configuration) |
| RSF, calendar clock | 0.591 | 0.666 | qualitatively agrees (calendar clock is much weaker) |
| AFT-Weibull, traffic clock | 0.693 | 0.832 | **diverges** — here the AFT nearly matches the RSF, unlike their result |

The qualitative conclusion that survives both datasets: **the clock (cumulative
traffic vs calendar time), not the model family, drives the headline number.** The AFT
divergence is plausibly the static-rate traffic proxy (below) plus NBI-vs-PennDOT
population and spell-construction differences — a constant-rate clock is a smooth
monotone transform of exposure that a linear AFT can exploit as easily as a forest,
whereas their accumulated time-varying ADTT is not.

**Feature mapping (their 12 covariates from NBI):** district:`HIGHWAY_DISTRICT_002`,
construction year:`YEAR_BUILT_027`, deck structure type:`DECK_STRUCTURE_TYPE_107`,
main material:`STRUCTURE_KIND_043A`, physical makeup/design:`STRUCTURE_TYPE_043B`,
span configuration:`MAIN_UNIT_SPANS_045`, membrane:`MEMBRANE_TYPE_108B`, wearing
surface:`SURFACE_TYPE_108A`, length:`STRUCTURE_LEN_MT_049`, deck width:`DECK_WIDTH_MT_052`,
ADT:`ADT_029`, ADTT:`ADT_029 × PERCENT_ADT_TRUCK_109 / 100`. Rebar type exists only in
PennDOT's BMS, not in NBI: documented limitation.

**Documented differences from the study:**
- Data source is the public NBI panel (1992–2025 annual snapshots), not PennDOT BMS
  inspection records; the state-owned population is matched via `MAINTENANCE_021 == 1`.
- One spell per bridge (its first observed CR-6 spell) with covariates frozen at spell
  start; spells already at CR 6 when the bridge enters the panel are excluded
  (left-censored — entry time unknown). Rating increases during a spell (rehab) censor
  the spell at the last year observed at CR 6.
- The cumulative-truck clock is ADTT-at-spell-start × years — a **static-rate proxy**
  (the study accumulated time-varying ADTT).

**Outcomes modeled** (each: RSF and AFT-Weibull, baseline features vs baseline+climate):
1. `spells_years` — CR 6 to 5 time-in-rating, calendar clock (their 0.591 comparison).
2. `spells_traffic` — CR 6 to 5 time-in-rating, cumulative-truck clock (their 0.836 / 0.693).
3. `deck_first_poor_age` — age at first deck rating ≤ 4 (this project's framework, deck only).
4. `composite_first_poor_age` — age at first deck/super/substructure ≤ 4 (the national
   pipeline's event) — answers whether the composite event is easier or harder than deck-only.

**Observation-year confound on the first-hit outcomes (established by the robustness
cell at the end):** those outcomes take covariates from the *kept* row, whose calendar
year differs systematically between events (first poor year) and censored bridges (last
survey, mostly 2025). Per-year climate values therefore let a model fingerprint the
observation era — target leakage that inflates the apparent climate gain (measured here:
roughly three quarters of the raw per-year climate delta). **Publication numbers for the
first-hit outcomes must come from the year-free climate-normals arms**, which carry only
spatial exposure signal. The spell outcomes freeze covariates at spell *entry*, where
event and censored spells overlap in calendar time, so they are far less exposed.
The final cell attaches a paired-bootstrap CI to the honest deltas: deck **+0.025**
(95% CI +0.019 to +0.031), composite **+0.023** (95% CI +0.018 to +0.027), both
p < 0.002 — statistically solid, ordinary-sized gains for adding one feature family.
(The national analogue of this leakage measurement, covering all three model families,
is `../Bridges_all_of_US/leakage_ablation_national.ipynb`.)

Set `STATE_FILTER = None` to extend the whole study to all 51 state files.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import concordance_index_censored
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index

NAT_DIR  = Path("../Bridges_all_of_US")          # inputs produced by the national pipeline
OUT_JSON = Path("us_pa_deck_study_results.json")

STATE_FILTER     = "42"    # FIPS of the state to study; None -> all downloaded states
STATE_OWNED_ONLY = True    # MAINTENANCE_021 == 1 (State Highway Agency), the study's population

KEYS = ["STATE_FIPS", "STRUCTURE_NUMBER_008"]    # structure # is unique only WITHIN a state
RARE_THRESHOLD = 50        # rare-level fold threshold, scaled to single-state sample sizes
TEST_SIZE, SEED = 0.2, 42
TRUCK_SCALE = 1e6          # cumulative-truck clock in millions of trucks (AFT numerics)

# Specialist capacity (state-scale samples): mirrors ma_bridges/RSF.ipynb at 300 trees.
# low_memory=True: only risk scores are needed, halves fit memory.
RSF_PARAMS = dict(n_estimators=300, min_samples_split=10, min_samples_leaf=5,
                  max_features="sqrt", low_memory=True, n_jobs=-1, random_state=SEED)
AFT_PENALIZER = 0.01       # ridge penalty; retried at 10x if the fit fails to converge

# Lu & Guler's covariates mapped to NBI (ADTT is derived below; rebar type unavailable).
BASE_NUM = ["YEAR_BUILT_027", "MAIN_UNIT_SPANS_045", "STRUCTURE_LEN_MT_049",
            "DECK_WIDTH_MT_052", "ADT_029", "ADTT"]
BASE_CAT = [("HIGHWAY_DISTRICT_002", "DISTRICT"), ("DECK_STRUCTURE_TYPE_107", "DECK_TYPE"),
            ("STRUCTURE_KIND_043A", "MATERIAL"), ("STRUCTURE_TYPE_043B", "DESIGN"),
            ("MEMBRANE_TYPE_108B", "MEMBRANE"), ("SURFACE_TYPE_108A", "SURFACE")]

PUBLISHED = {"RSF, traffic clock": 0.8362, "RSF, calendar clock": 0.5913,
             "AFT-Weibull, traffic clock": 0.6931}

print(f"state={STATE_FILTER}   state-owned only={STATE_OWNED_ONLY}   "
      f"baseline features: {len(BASE_NUM)} numeric + {len(BASE_CAT)} categorical")

In [ ]:
# ── Load the bridge-year panel and extract deck CR-6 spells ───────────────────
RAW_FEATURES = [c for c, _ in BASE_CAT] + ["MAIN_UNIT_SPANS_045", "STRUCTURE_LEN_MT_049",
                "DECK_WIDTH_MT_052", "ADT_029", "PERCENT_ADT_TRUCK_109", "YEAR_BUILT_027"]
USECOLS = KEYS + ["SURVEY_YEAR", "DECK_COND_058", "SUPERSTRUCTURE_COND_059",
                  "SUBSTRUCTURE_COND_060", "MAINTENANCE_021"] + RAW_FEATURES
STR_COLS = {c: str for c in KEYS + ["DECK_COND_058", "SUPERSTRUCTURE_COND_059",
                                    "SUBSTRUCTURE_COND_060"]}

files = (sorted((NAT_DIR / "raw").glob("*_bridges.csv")) if STATE_FILTER is None
         else [f for f in (NAT_DIR / "raw").glob("*_bridges.csv")
               if pd.read_csv(f, usecols=["STATE_FIPS"], nrows=1, dtype=str)
                    ["STATE_FIPS"].iloc[0].strip() == STATE_FILTER])
assert files, f"no raw state file found for FIPS {STATE_FILTER}"

# low_memory=False: pandas 3.0's block-wise parse + usecols raises IndexError while
# composing its mixed-dtype warning (same bug the national Notebook 4 (build_national_rsf_dataset.ipynb) works around).
panel = pd.concat([pd.read_csv(f, usecols=USECOLS, dtype=STR_COLS, low_memory=False)
                   for f in files], ignore_index=True)
for k in KEYS:
    panel[k] = panel[k].str.strip()
if STATE_OWNED_ONLY:
    panel = panel[pd.to_numeric(panel["MAINTENANCE_021"], errors="coerce") == 1]

panel["SURVEY_YEAR"] = pd.to_numeric(panel["SURVEY_YEAR"], errors="coerce").astype("int32")
# Spell clock uses the raw deck rating; 'N' (not applicable, e.g. culverts) -> NaN.
panel["deck_rc"] = pd.to_numeric(panel["DECK_COND_058"].replace("N", np.nan), errors="coerce")
panel = panel.sort_values(KEYS + ["SURVEY_YEAR"], kind="mergesort").reset_index(drop=True)
print(f"panel: {len(panel):,} bridge-year rows, {panel.groupby(KEYS).ngroups:,} bridges")

# First CR-6 spell per bridge. Entry must be OBSERVED (previous rating known and != 6),
# excluding left-censored spells. Event = rating drops to <=5; a rating rise above 6
# (rehab) censors at the last year seen at 6; otherwise censored at panel end.
idx, events, durs = [], [], []
for _, g in panel.groupby(KEYS, sort=False):
    rc, yr, gi = g["deck_rc"].to_numpy(), g["SURVEY_YEAR"].to_numpy(), g.index.to_numpy()
    entry = None
    for i in range(1, len(rc)):
        if rc[i] == 6 and not np.isnan(rc[i - 1]) and rc[i - 1] != 6:
            entry = i
            break
    if entry is None:
        continue
    event, end, last6 = 0, None, yr[entry]
    for j in range(entry + 1, len(rc)):
        if np.isnan(rc[j]):
            continue
        if rc[j] <= 5:
            event, end = 1, yr[j]
            break
        if rc[j] > 6:
            end = last6
            break
        last6 = yr[j]
    dur = float((end if end is not None else last6) - yr[entry])
    if dur <= 0:                       # entered CR 6 with no usable follow-up
        continue
    idx.append(gi[entry]); events.append(event); durs.append(dur)

spells = panel.loc[idx].copy()
spells["event"] = np.array(events, dtype="int8")
spells["dur_years"] = durs
spells["ADTT"] = (pd.to_numeric(spells["ADT_029"], errors="coerce")
                  * pd.to_numeric(spells["PERCENT_ADT_TRUCK_109"], errors="coerce") / 100)
spells["dur_trucks"] = spells["ADTT"] * 365.25 * spells["dur_years"] / TRUCK_SCALE

assert len(spells) > 5_000, f"only {len(spells)} spells extracted"
assert (spells["dur_years"] > 0).all()
er = spells["event"].mean()
assert 0.05 < er < 0.90, f"implausible spell event rate {er:.2f}"
print(f"spells: {len(spells):,}   events: {int(spells['event'].sum()):,} ({er*100:.1f}%)   "
      f"median duration: {spells['dur_years'].median():.0f} yr   "
      f"valid truck clock: {(spells['dur_trucks'] > 0).sum():,}")

In [ ]:
# ── First-hit-poor datasets (deck-only and composite), Option-A collapse ──────
# Same semantics as the national Notebook 4 (build_national_rsf_dataset.ipynb) / ../ma_bridges/Build_RSF.ipynb: one row per
# bridge = first event row, else last observed row; time = bridge age at that row.
def cond_num(col):
    """'N' (not applicable) -> 10 so it can never read as poor — the national build notebook's convention."""
    return pd.to_numeric(panel[col].replace("N", 10), errors="coerce")

deck_n, sup_n, sub_n = (cond_num(c) for c in
    ("DECK_COND_058", "SUPERSTRUCTURE_COND_059", "SUBSTRUCTURE_COND_060"))
panel["deck_poor"] = (deck_n <= 4).astype("int8")
panel["comp_poor"] = ((deck_n <= 4) | (sup_n <= 4) | (sub_n <= 4)).astype("int8")
panel["bridge_age"] = panel["SURVEY_YEAR"] - pd.to_numeric(panel["YEAR_BUILT_027"],
                                                           errors="coerce")

def collapse_first_hit(event_col):
    ev = panel[panel[event_col] == 1].drop_duplicates(KEYS, keep="first")
    last = panel.drop_duplicates(KEYS, keep="last")
    cens = last.merge(ev[KEYS], on=KEYS, how="left", indicator=True)
    cens = cens[cens["_merge"] == "left_only"].drop(columns="_merge")
    out = pd.concat([ev, cens], ignore_index=True)
    out = out.rename(columns={event_col: "event"})
    out["event"] = out["event"].astype("int8")
    out["time"] = out["bridge_age"].astype(float)
    out = out[out["time"] > 0].copy()
    out["ADTT"] = (pd.to_numeric(out["ADT_029"], errors="coerce")
                   * pd.to_numeric(out["PERCENT_ADT_TRUCK_109"], errors="coerce") / 100)
    return out

deck_fh = collapse_first_hit("deck_poor")
comp_fh = collapse_first_hit("comp_poor")
# The composite event contains the deck event, so its rate must be >= deck-only's.
assert comp_fh["event"].mean() >= deck_fh["event"].mean()
print(f"deck first-hit:      {len(deck_fh):,} bridges, {deck_fh['event'].mean()*100:.1f}% events")
print(f"composite first-hit: {len(comp_fh):,} bridges, {comp_fh['event'].mean()*100:.1f}% events")

In [ ]:
# ── Climate merge + one-hot encoding ──────────────────────────────────────────
cell_map = pd.read_csv(NAT_DIR / "us_bridge_cell_map.csv",
                       dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str})
coastal  = pd.read_csv(NAT_DIR / "us_bridge_coastal_distance.csv",
                       dtype={"STRUCTURE_NUMBER_008": str, "STATE_FIPS": str})
for d in (cell_map, coastal):
    for k in KEYS:
        d[k] = d[k].str.strip()
cell_map = cell_map.drop_duplicates(KEYS, keep="first")
coastal  = coastal.drop_duplicates(KEYS, keep="first")

CLIMATE_FEATURES = [c for c in pd.read_csv(NAT_DIR / "us_climate_by_cell_year.csv",
                                           nrows=0).columns
                    if c not in ("SURVEY_YEAR", "grid_lat", "grid_lon")]
_dt = {c: "float32" for c in CLIMATE_FEATURES}
_dt.update({"SURVEY_YEAR": "int32", "grid_lat": "float64", "grid_lon": "float64"})
climate = pd.read_csv(NAT_DIR / "us_climate_by_cell_year.csv", dtype=_dt)
CLIM_COLS = CLIMATE_FEATURES + ["dist_to_coast_km"]
print(f"climate: {len(climate):,} cell-years, {len(CLIMATE_FEATURES)} features")

def norm_code(series):
    """Collapse float/string spellings of the same NBI code ('2.0'/'2' -> '2') so one-hot
    cannot split a level — ported from Notebook 4 (build_national_rsf_dataset.ipynb)."""
    s = series.astype(str).str.strip()
    num = pd.to_numeric(s, errors="coerce")
    is_int = num.notna() & (num == np.floor(num))
    int_str = num.round().astype("Int64").astype("string")
    return s.mask(is_int, int_str)

def encode(df):
    """Baseline+climate design matrix. Returns (X, y-parts, baseline column list,
    reference dummies to drop for the AFT full-rank design)."""
    df = (df.merge(cell_map, on=KEYS, how="left", validate="m:1")
            .merge(climate, on=["grid_lat", "grid_lon", "SURVEY_YEAR"], how="left",
                   validate="m:1")
            .merge(coastal[KEYS + ["dist_to_coast_km"]], on=KEYS, how="left",
                   validate="m:1"))
    first_dummies, parts = [], []
    for col, pfx in BASE_CAT:
        s = norm_code(df[col]).astype(str).str.strip().replace(
            {"nan": "Unknown", "None": "Unknown", "N": "Unknown", "<NA>": "Unknown"})
        vc = s.value_counts()
        s = s.where(~s.isin(vc[vc < RARE_THRESHOLD].index.difference(["Unknown"])), "Other")
        dummies = pd.get_dummies(s, prefix=pfx, dtype="int8")
        first_dummies.append(dummies.columns[0])
        parts.append(dummies)
    num = df[BASE_NUM + CLIM_COLS].apply(pd.to_numeric, errors="coerce")
    X = pd.concat(parts + [num.reset_index(drop=True)], axis=1)
    X.index = df.index
    baseline_cols = [c for c in X.columns if c not in CLIM_COLS]
    return X, baseline_cols, first_dummies

X_sp, base_sp, fd_sp = encode(spells.reset_index(drop=True))
X_dk, base_dk, fd_dk = encode(deck_fh.reset_index(drop=True))
X_cp, base_cp, fd_cp = encode(comp_fh.reset_index(drop=True))
print(f"encoded: spells {X_sp.shape}   deck {X_dk.shape}   composite {X_cp.shape}   "
      f"(baseline = all minus {len(CLIM_COLS)} climate cols)")

In [ ]:
# ── RSF arms: baseline vs baseline+climate for every outcome ──────────────────
def rsf_pair(X, event, dur, baseline_cols):
    """Fit RSF twice on one shared split (paired arms): baseline features, then all."""
    keep = pd.to_numeric(dur, errors="coerce") > 0
    Xk, ek, dk = X[keep], event[keep].astype(bool).to_numpy(), dur[keep].to_numpy(float)
    y = Surv.from_arrays(event=ek, time=dk)
    Xtr, Xte, ytr, yte = train_test_split(Xk, y, test_size=TEST_SIZE, random_state=SEED)
    out = {"n": int(len(Xk)), "n_test": int(len(Xte)), "event_rate": round(float(ek.mean()), 3)}
    for label, cols in [("baseline", baseline_cols), ("climate", list(X.columns))]:
        t0 = time.time()
        rsf = RandomSurvivalForest(**RSF_PARAMS).fit(Xtr[cols], ytr)
        ci = concordance_index_censored(yte["event"], yte["time"], rsf.predict(Xte[cols]))[0]
        out[f"rsf_{label}"] = round(float(ci), 4)
        print(f"    RSF {label:<8} C-index {ci:.4f}  ({time.time()-t0:.0f}s)", flush=True)
    out["rsf_climate_delta"] = round(out["rsf_climate"] - out["rsf_baseline"], 4)
    return out

results = {}
sp = spells.reset_index(drop=True)
for name, X, base, ev, dur in [
    ("spells_years",   X_sp, base_sp, sp["event"], sp["dur_years"]),
    ("spells_traffic", X_sp, base_sp, sp["event"], sp["dur_trucks"]),
    ("deck_first_poor_age",      X_dk, base_dk,
        deck_fh.reset_index(drop=True)["event"], deck_fh.reset_index(drop=True)["time"]),
    ("composite_first_poor_age", X_cp, base_cp,
        comp_fh.reset_index(drop=True)["event"], comp_fh.reset_index(drop=True)["time"]),
]:
    print(f"[{name}]", flush=True)
    results[name] = rsf_pair(X, ev, dur, base)

In [ ]:
# ── AFT-Weibull arms on the same splits (full-rank drop-first design) ─────────
def aft_pair(name, X, event, dur, baseline_cols, first_dummies):
    keep = pd.to_numeric(dur, errors="coerce") > 0
    Xk = X[keep].drop(columns=first_dummies)      # reference dummies out -> full rank
    ek, dk = event[keep].astype("int8").to_numpy(), dur[keep].to_numpy(float)
    tr, te = train_test_split(np.arange(len(Xk)), test_size=TEST_SIZE, random_state=SEED)
    base = [c for c in baseline_cols if c in Xk.columns]
    for label, cols in [("baseline", base), ("climate", list(Xk.columns))]:
        M = Xk[cols].copy()
        # AFT cannot take NaN: flag missingness, median-impute (train medians), then
        # standardise on train statistics so the Newton solver is well-conditioned.
        for c in M.columns[M.isna().any()].tolist():
            M[f"{c}_missing"] = M[c].isna().astype("int8")
        med = M.iloc[tr].median()
        M = M.fillna(med)
        mu, sd = M.iloc[tr].mean(), M.iloc[tr].std()
        good = sd[sd > 0].index                    # zero-variance columns break the solver
        M = (M[good] - mu[good]) / sd[good]
        M["T"], M["E"] = dk, ek
        train, test = M.iloc[tr], M.iloc[te]
        try:
            aft = WeibullAFTFitter(penalizer=AFT_PENALIZER)
            aft.fit(train, duration_col="T", event_col="E")
        except Exception:
            aft = WeibullAFTFitter(penalizer=AFT_PENALIZER * 10)
            aft.fit(train, duration_col="T", event_col="E")
        pred = aft.predict_median(test.drop(columns=["T", "E"]))
        ci = concordance_index(test["T"], pred, test["E"])
        results[name][f"aft_{label}"] = round(float(ci), 4)
        print(f"    AFT {label:<8} C-index {ci:.4f}", flush=True)
    results[name]["aft_climate_delta"] = round(
        results[name]["aft_climate"] - results[name]["aft_baseline"], 4)

for name, X, base, fd, ev, dur in [
    ("spells_years",   X_sp, base_sp, fd_sp, sp["event"], sp["dur_years"]),
    ("spells_traffic", X_sp, base_sp, fd_sp, sp["event"], sp["dur_trucks"]),
    ("deck_first_poor_age",      X_dk, base_dk, fd_dk,
        deck_fh.reset_index(drop=True)["event"], deck_fh.reset_index(drop=True)["time"]),
    ("composite_first_poor_age", X_cp, base_cp, fd_cp,
        comp_fh.reset_index(drop=True)["event"], comp_fh.reset_index(drop=True)["time"]),
]:
    print(f"[{name}]", flush=True)
    aft_pair(name, X, ev, dur, base, fd)

In [ ]:
# ── Results table: us_pa_deck_study_results.json ────────────────────────────
out = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "design": {
        "state_fips": STATE_FILTER, "state_owned_only": STATE_OWNED_ONLY,
        "spells": int(len(spells)), "spell_event_rate": round(float(spells["event"].mean()), 3),
        "rsf_params": {k: (v if isinstance(v, (int, float, str, bool, type(None)))
                           else str(v)) for k, v in RSF_PARAMS.items()},
        "aft_penalizer": AFT_PENALIZER, "test_size": TEST_SIZE, "seed": SEED,
        "truck_clock_units": f"trucks / {TRUCK_SCALE:.0e}",
        "limitations": ["rebar type not in NBI", "one spell per bridge, covariates at spell start",
                        "left-censored spells excluded", "truck clock = ADTT at spell start x years"],
    },
    "published_lu_guler_2022": PUBLISHED,
    "results": results,
}
OUT_JSON.write_text(json.dumps(out, indent=2))
print(f"saved {OUT_JSON}\n")

table = pd.DataFrame(results).T[["n", "event_rate", "rsf_baseline", "rsf_climate",
                                 "rsf_climate_delta", "aft_baseline", "aft_climate",
                                 "aft_climate_delta"]]
print(table.to_string())
print("\nLu & Guler 2022 (test):", ", ".join(f"{k} = {v}" for k, v in PUBLISHED.items()))
print("\ncomposite vs deck-only (RSF, +climate): "
      f"{results['composite_first_poor_age']['rsf_climate']:.4f} vs "
      f"{results['deck_first_poor_age']['rsf_climate']:.4f}")

for name, r in results.items():
    for m in ("rsf_baseline", "rsf_climate", "aft_baseline", "aft_climate"):
        assert 0.0 < r[m] < 1.0, f"{name}/{m} out of range"
print("\nAll arms completed.")

In [ ]:
# ── Robustness: year-free climate NORMALS on the first-hit outcomes ───────────
# The per-year climate merge keys on the KEPT row's SURVEY_YEAR, and kept rows come from
# systematically different calendar years for events (first poor year) than for censored
# bridges (last survey, mostly 2025), year-specific climate values let the model
# fingerprint the observation era, inflating the apparent climate gain. 1992–2025
# per-cell means carry only spatial exposure; the surviving delta is the honest climate
# contribution and is what the publication should report for these outcomes.
normals = (climate.groupby(["grid_lat", "grid_lon"], as_index=False)[CLIMATE_FEATURES]
                  .mean())
print(f"normals: {len(normals):,} grid cells")

def encode_normals(df):
    """encode() with the per-year climate merge replaced by year-free normals."""
    df = (df.merge(cell_map, on=KEYS, how="left", validate="m:1")
            .merge(normals, on=["grid_lat", "grid_lon"], how="left")
            .merge(coastal[KEYS + ["dist_to_coast_km"]], on=KEYS, how="left",
                   validate="m:1"))
    parts = []
    for col, pfx in BASE_CAT:
        s = norm_code(df[col]).astype(str).str.strip().replace(
            {"nan": "Unknown", "None": "Unknown", "N": "Unknown", "<NA>": "Unknown"})
        vc = s.value_counts()
        s = s.where(~s.isin(vc[vc < RARE_THRESHOLD].index.difference(["Unknown"])), "Other")
        parts.append(pd.get_dummies(s, prefix=pfx, dtype="int8"))
    num = df[BASE_NUM + CLIM_COLS].apply(pd.to_numeric, errors="coerce")
    X = pd.concat(parts + [num.reset_index(drop=True)], axis=1)
    X.index = df.index
    return X

# Baselines come from the saved results file so this cell is re-runnable after a kernel
# restart (only cells 2–5 are required in memory).
saved = json.loads(OUT_JSON.read_text())
robust, normals_pred = {}, {}
for name, df_src in [("deck_first_poor_age", deck_fh), ("composite_first_poor_age", comp_fh)]:
    df = df_src.reset_index(drop=True)
    Xn = encode_normals(df)
    keep = pd.to_numeric(df["time"], errors="coerce") > 0
    y = Surv.from_arrays(event=df["event"][keep].astype(bool).to_numpy(),
                         time=df["time"][keep].to_numpy(float))
    Xtr, Xte, ytr, yte = train_test_split(Xn[keep], y, test_size=TEST_SIZE,
                                          random_state=SEED)
    t0 = time.time()
    rsf = RandomSurvivalForest(**RSF_PARAMS).fit(Xtr, ytr)
    pred = rsf.predict(Xte)
    ci = concordance_index_censored(yte["event"], yte["time"], pred)[0]
    normals_pred[name] = pred, yte   # reused by the significance cell below
    r = saved["results"][name]
    robust[name] = {
        "rsf_climate_normals": round(float(ci), 4),
        "normals_delta_vs_baseline": round(float(ci) - r["rsf_baseline"], 4),
        "per_year_delta_vs_baseline": r["rsf_climate_delta"],
    }
    print(f"[{name}] normals C-index {ci:.4f}   "
          f"(baseline {r['rsf_baseline']:.4f}, per-year {r['rsf_climate']:.4f})   "
          f"({time.time()-t0:.0f}s)", flush=True)

saved["robustness_climate_normals"] = {
    "rationale": "per-year climate keyed on the kept row's SURVEY_YEAR leaks the "
                 "observation era on first-hit outcomes; normals (1992-2025 cell means) "
                 "carry only spatial exposure — publication numbers for first-hit "
                 "outcomes should use the normals deltas",
    **robust,
}
OUT_JSON.write_text(json.dumps(saved, indent=2))
print(f"\nupdated {OUT_JSON}")
for name, r in robust.items():
    leak_share = 1 - r["normals_delta_vs_baseline"] / r["per_year_delta_vs_baseline"]
    print(f"{name}: honest climate delta {r['normals_delta_vs_baseline']:+.4f}   "
          f"({leak_share*100:.0f}% of the per-year delta was observation-year signal)")

In [ ]:
# ── Significance: paired bootstrap CI for the honest (normals) climate delta ──
# Requires the robustness cell above (uses its stashed normals-arm test predictions).
# The baseline arms are re-fit on the identical split — same SEED/TEST_SIZE on the same
# rows reproduces the split exactly, which the asserts below verify against the saved
# JSON. Both models are then scored on the SAME resampled test rows, so the CI is for
# the paired difference — the number that belongs next to the point estimate in the
# paper. Resamples are scored with lifelines' O(n log n) concordance (sksurv's exact
# implementation is too slow for 2,000 calls); agreement between the two is asserted.
B = 1000
rng = np.random.default_rng(2026)
saved = json.loads(OUT_JSON.read_text())
signif = {}
for name, df_src, X_yearly, base_cols in [
        ("deck_first_poor_age", deck_fh, X_dk, base_dk),
        ("composite_first_poor_age", comp_fh, X_cp, base_cp)]:
    df = df_src.reset_index(drop=True)
    keep = pd.to_numeric(df["time"], errors="coerce") > 0
    y = Surv.from_arrays(event=df["event"][keep].astype(bool).to_numpy(),
                         time=df["time"][keep].to_numpy(float))
    Xtr, Xte, ytr, yte = train_test_split(X_yearly[keep][base_cols], y,
                                          test_size=TEST_SIZE, random_state=SEED)
    t0 = time.time()
    pred_b = RandomSurvivalForest(**RSF_PARAMS).fit(Xtr, ytr).predict(Xte)
    pred_n, yte_n = normals_pred[name]
    ci_b = concordance_index_censored(yte["event"], yte["time"], pred_b)[0]
    ci_n = concordance_index_censored(yte["event"], yte["time"], pred_n)[0]
    # both arms must sit on the same test rows, and both point estimates must
    # reproduce what the results JSON already reports
    assert np.array_equal(yte["time"], yte_n["time"])
    assert np.array_equal(yte["event"], yte_n["event"])
    assert abs(ci_b - saved["results"][name]["rsf_baseline"]) < 1e-3
    assert abs(ci_n - saved["robustness_climate_normals"][name]["rsf_climate_normals"]) < 1e-3
    # lifelines scores survival (higher = longer), RSF scores risk -> negate predictions
    assert abs(concordance_index(yte["time"], -pred_b, yte["event"]) - ci_b) < 2e-3
    assert abs(concordance_index(yte["time"], -pred_n, yte["event"]) - ci_n) < 2e-3

    deltas = np.empty(B)
    for b in range(B):
        ii = rng.integers(0, len(yte), len(yte))
        t_i, e_i = yte["time"][ii], yte["event"][ii]
        deltas[b] = (concordance_index(t_i, -pred_n[ii], e_i)
                     - concordance_index(t_i, -pred_b[ii], e_i))
    lo, hi = np.percentile(deltas, [2.5, 97.5])
    p_two = 2 * min(float((deltas <= 0).mean()), float((deltas >= 0).mean()))
    signif[name] = {
        "delta_point": round(float(ci_n - ci_b), 4),
        "delta_ci95": [round(float(lo), 4), round(float(hi), 4)],
        "p_two_sided": f"< {2 / B:g}" if p_two == 0 else round(p_two, 4),
        "bootstrap_B": B,
        "n_test": int(len(yte)),
        "n_test_events": int(yte["event"].sum()),
    }
    print(f"[{name}] climate delta {ci_n - ci_b:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]   "
          f"p {signif[name]['p_two_sided']}   ({time.time()-t0:.0f}s)", flush=True)

saved["significance_paired_bootstrap"] = {
    "method": "baseline vs climate-normals RSF, both scored on the same resampled rows "
              "of the shared test split; percentile CI over B bootstrap resamples",
    **signif,
}
OUT_JSON.write_text(json.dumps(saved, indent=2))
print(f"\nupdated {OUT_JSON}")